# LTX-2.3 Hindi Poem — Kaggle Dual-T4 Runner
Fresh Kaggle setup, two independent GPU workers, resumable segment generation, and exact original-audio muxing.

**Important:** This notebook runs a ComfyUI **API-format prompt JSON** exported from your optimized LTX workflow. The uploaded `Workflow-2.json` is a UI-format graph; export its API prompt from ComfyUI (Save (API Format)) and provide that file below. The notebook validates node classes and configured input-node IDs before rendering. It does not pretend two T4 cards share VRAM: each worker loads its own model copy on one GPU.

In [ ]:
# 1. Kaggle configuration — edit these values or set Kaggle Secrets / environment variables.
from pathlib import Path
import os, json, subprocess, sys, time, shutil, socket, threading, queue, hashlib, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed
ROOT = Path('/kaggle/working/ltx_hindi')
COMFY_REPO = os.getenv('COMFY_REPO', 'https://github.com/comfyanonymous/ComfyUI.git')
COMFY_ROOT = ROOT / 'ComfyUI'
INPUT_DIR = Path('/kaggle/input')
OUTPUT_DIR = ROOT / 'output'; OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR = ROOT / 'work'; WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = ROOT / 'checkpoints'; CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
API_WORKFLOW = os.getenv('LTX_API_WORKFLOW', str(INPUT_DIR / 'ltx-api-workflow.json'))
IMAGE_PATH = os.getenv('LTX_IMAGE_PATH', str(INPUT_DIR / 'dada.png'))
AUDIO_PATH = os.getenv('LTX_AUDIO_PATH', '')  # set to Kaggle input audio path
SEGMENT_SECONDS = int(os.getenv('LTX_SEGMENT_SECONDS', '5'))
FPS = int(os.getenv('LTX_FPS', '24'))
WIDTH = int(os.getenv('LTX_WIDTH', '864')); HEIGHT = int(os.getenv('LTX_HEIGHT', '480'))
GPU_IDS = [int(x) for x in os.getenv('LTX_GPU_IDS', '0,1').split(',') if x.strip()]
MAX_WORKERS = min(2, len(GPU_IDS))
HF_TOKEN = os.getenv('HF_TOKEN', '')  # preferably add Kaggle Secret named HF_TOKEN
COMFY_PORTS = [int(x) for x in os.getenv('LTX_COMFY_PORTS', '8188,8189').split(',')]
print('Root:', ROOT, '| GPU IDs:', GPU_IDS, '| workers:', MAX_WORKERS)


In [ ]:
# 2. Fresh runtime and hardware validation
assert shutil.which('ffmpeg') or True
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg','git','aria2'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','requests','soundfile','numpy','Pillow'], check=True)
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Kaggle GPU is not enabled. Set Notebook Accelerator to GPU.')
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i); print(i, p.name, round(p.total_memory/1024**3,2),'GiB')
if len(GPU_IDS)<1 or max(GPU_IDS)>=torch.cuda.device_count(): raise RuntimeError(f'Invalid GPU_IDS={GPU_IDS}; visible GPUs={torch.cuda.device_count()}')
if len(GPU_IDS)<2: print('WARNING: only one GPU visible; execution will use one worker.')


In [ ]:
# 3. Install ComfyUI and configurable custom nodes
if not COMFY_ROOT.exists(): subprocess.run(['git','clone','--depth','1',COMFY_REPO,str(COMFY_ROOT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(COMFY_ROOT/'requirements.txt')],check=True)
CUSTOM_NODES = [x.strip() for x in os.getenv('LTX_CUSTOM_NODE_REPOS','').splitlines() if x.strip()]
CUSTOM_ROOT=COMFY_ROOT/'custom_nodes'; CUSTOM_ROOT.mkdir(exist_ok=True)
for repo in CUSTOM_NODES:
    name=repo.rstrip('/').split('/')[-1].removesuffix('.git'); dest=CUSTOM_ROOT/name
    if not dest.exists(): subprocess.run(['git','clone','--depth','1',repo,str(dest)],check=True)
    req=dest/'requirements.txt'
    if req.exists(): subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(req)],check=False)
print('ComfyUI installed:', COMFY_ROOT)
print('Set LTX_CUSTOM_NODE_REPOS as newline-separated git URLs for all custom nodes required by your graph.')


In [ ]:
# 4. Model storage and validation
# Place model files in Kaggle Dataset or mounted Drive; provide model paths as environment variables.
MODEL_PATHS = {k:os.getenv(k,'') for k in ['LTX_VIDEO_MODEL','LTX_TEXT_ENCODER','LTX_VIDEO_VAE','LTX_AUDIO_VAE','LTX_TALKING_HEAD_LORA']}
for key,path in MODEL_PATHS.items():
    if path:
        p=Path(path).expanduser()
        if not p.exists() or not p.is_file(): raise FileNotFoundError(f'{key} does not exist: {p}')
        print(key, p, f'{p.stat().st_size/1024**3:.2f} GiB')
    else: print(key, '(not configured; ensure model is already installed in ComfyUI model folders)')
if HF_TOKEN: os.environ['HF_TOKEN']=HF_TOKEN
print('Do not put tokens in notebook cells. Use Kaggle Secrets or environment variables.')


In [ ]:
# 5. Validate inputs and inspect API workflow
import requests
image_path=Path(IMAGE_PATH); audio_path=Path(AUDIO_PATH) if AUDIO_PATH else None
if not image_path.is_file(): raise FileNotFoundError(f'Reference image not found: {image_path}')
if audio_path is None or not audio_path.is_file(): raise FileNotFoundError('Set LTX_AUDIO_PATH to your Hindi WAV/MP3 file in Kaggle input.')
wf_path=Path(API_WORKFLOW)
if not wf_path.is_file(): raise FileNotFoundError('Provide API-format prompt JSON as ltx-api-workflow.json in a Kaggle input dataset; export via ComfyUI > Save (API Format).')
api_prompt=json.loads(wf_path.read_text(encoding='utf-8'))
if 'prompt' in api_prompt and isinstance(api_prompt['prompt'],dict): api_prompt=api_prompt['prompt']
if not isinstance(api_prompt,dict) or not api_prompt: raise ValueError('Workflow JSON must be a non-empty ComfyUI API prompt object.')
print('API workflow nodes:',len(api_prompt)); print('Classes:',sorted(set(v.get('class_type','?') for v in api_prompt.values())))
required=['LoadImage','LoadAudio']
classes={v.get('class_type') for v in api_prompt.values()}
print('Standard image/audio loaders present:', {x:x in classes for x in required})


In [ ]:
# 6. Audio analysis and exact-duration segmentation
def run(cmd): return subprocess.run(cmd,check=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
probe=json.loads(run(['ffprobe','-v','error','-show_entries','format=duration','-of','json',str(audio_path)]).stdout)
duration=float(probe['format']['duration']); print('Audio duration:',round(duration,3),'seconds')
segments=[]; start=0.0; idx=0
while start < duration-1e-3:
    dur=min(float(SEGMENT_SECONDS),duration-start); target=WORK_DIR/f'audio_{idx:05d}.wav'
    if not target.exists():
        run(['ffmpeg','-y','-ss',str(start),'-i',str(audio_path),'-t',str(dur),'-vn','-ac','2','-ar','24000','-c:a','pcm_s16le',str(target)])
    segments.append({'index':idx,'start':start,'duration':dur,'audio':str(target)})
    start+=dur; idx+=1
print('Segments:',len(segments))


In [ ]:
# 7. ComfyUI worker utilities — one independent server/model copy per GPU
def free_port(port):
    s=socket.socket(); ok=s.connect_ex(('127.0.0.1',port))!=0; s.close(); return ok
def wait_server(port,proc,timeout=900):
    deadline=time.time()+timeout
    while time.time()<deadline:
        if proc.poll() is not None: raise RuntimeError(f'ComfyUI on port {port} exited with code {proc.returncode}')
        try:
            if requests.get(f'http://127.0.0.1:{port}/system_stats',timeout=3).ok:return
        except requests.RequestException: pass
        time.sleep(3)
    raise TimeoutError(f'ComfyUI on port {port} did not become ready')
def start_worker(gpu,port):
    if not free_port(port): raise RuntimeError(f'Port {port} already in use')
    env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(gpu)
    log=open(ROOT/f'comfy_gpu{gpu}.log','w')
    proc=subprocess.Popen([sys.executable,str(COMFY_ROOT/'main.py'),'--listen','127.0.0.1','--port',str(port),'--disable-auto-launch'],cwd=COMFY_ROOT,env=env,stdout=log,stderr=subprocess.STDOUT)
    wait_server(port,proc); return {'gpu':gpu,'port':port,'proc':proc,'log':log,'url':f'http://127.0.0.1:{port}'}
workers=[]
for gpu,port in zip(GPU_IDS[:MAX_WORKERS],COMFY_PORTS[:MAX_WORKERS]):
    workers.append(start_worker(gpu,port)); print('Ready:',gpu,port)


In [ ]:
# 8. Workflow adapter — configure node IDs/fields to match your exported LTX API prompt
NODE_MAP = json.loads(os.getenv('LTX_NODE_MAP_JSON','{}'))
# Example NODE_MAP: {"image":{"id":"12","field":"image"},"audio":{"id":"15","field":"audio"},"duration":{"id":"23","field":"duration"},"seed":{"id":"44","field":"seed"}}
def upload_input(worker, path):
    with open(path,'rb') as f:
        r=requests.post(worker['url']+'/upload/image',files={'image':(Path(path).name,f)},data={'type':'input','overwrite':'true'},timeout=300)
    r.raise_for_status(); return r.json()['name']
def make_prompt(worker, seg, seed):
    p=json.loads(json.dumps(api_prompt))
    for kind, local in [('image',str(image_path)),('audio',seg['audio'])]:
        cfg=NODE_MAP.get(kind)
        if not cfg: continue
        node=p[str(cfg['id'])]; name=upload_input(worker,local); node['inputs'][cfg['field']]=name
    for kind,value in [('duration',seg['duration']),('seed',seed)]:
        cfg=NODE_MAP.get(kind)
        if cfg: p[str(cfg['id'])]['inputs'][cfg['field']]=value
    return p
print('Node map:',NODE_MAP or 'NOT CONFIGURED — set LTX_NODE_MAP_JSON to map input nodes')


In [ ]:
# 9. Submit, poll, download and checkpoint
def submit_and_wait(worker,prompt,timeout=3600):
    r=requests.post(worker['url']+'/prompt',json={'prompt':prompt},timeout=120); r.raise_for_status(); pid=r.json()['prompt_id']
    deadline=time.time()+timeout
    while time.time()<deadline:
        h=requests.get(worker['url']+'/history/'+pid,timeout=30); h.raise_for_status(); hist=h.json().get(pid,{})
        if hist.get('outputs'): return pid,hist['outputs']
        if hist.get('status',{}).get('status_str')=='error': raise RuntimeError(f'ComfyUI job failed: {hist}')
        time.sleep(4)
    raise TimeoutError(f'ComfyUI prompt {pid} timed out')
def download_outputs(worker,outputs,dest):
    saved=[]
    for node_id,entry in outputs.items():
        for kind in ('videos','gifs','images'):
            for item in entry.get(kind,[]):
                params={'filename':item['filename'],'subfolder':item.get('subfolder',''),'type':item.get('type','output')}
                data=requests.get(worker['url']+'/view',params=params,timeout=300); data.raise_for_status()
                suffix=Path(item['filename']).suffix or ('.mp4' if kind in ('videos','gifs') else '.png')
                target=dest/f'node_{node_id}_{len(saved)}{suffix}'; target.write_bytes(data.content); saved.append(target)
    return saved
def checkpoint(seg_idx, files, prompt_id, gpu):
    rec={'segment':seg_idx,'files':[str(x) for x in files],'prompt_id':prompt_id,'gpu':gpu,'completed_at':time.time()}
    (CHECKPOINT_DIR/f'{seg_idx:05d}.json').write_text(json.dumps(rec,indent=2))
def completed(seg_idx):
    f=CHECKPOINT_DIR/f'{seg_idx:05d}.json'
    if not f.exists(): return None
    rec=json.loads(f.read_text()); paths=[Path(x) for x in rec.get('files',[])]
    return paths if paths and all(x.exists() and x.stat().st_size>0 for x in paths) else None


In [ ]:
# 10. Parallel segment execution — each worker owns a separate T4 and separate ComfyUI process
if not NODE_MAP.get('image') or not NODE_MAP.get('audio'):
    raise RuntimeError('Configure LTX_NODE_MAP_JSON with the image and audio loader node IDs/fields before running generation.')
def render_one(seg,worker):
    idx=seg['index']; old=completed(idx)
    if old: return idx,old,'resumed'
    prompt=make_prompt(worker,seg,seed=10000+idx)
    pid,outs=submit_and_wait(worker,prompt)
    files=download_outputs(worker,outs,OUTPUT_DIR/f'segment_{idx:05d}') if False else None
    segdir=OUTPUT_DIR/f'segment_{idx:05d}'; segdir.mkdir(exist_ok=True)
    files=download_outputs(worker,outs,segdir)
    if not files: raise RuntimeError(f'No downloadable outputs for segment {idx}; check SaveVideo/SaveImage node outputs.')
    checkpoint(idx,files,pid,worker['gpu']); return idx,files,'generated'
results={}; failures=[]
with ThreadPoolExecutor(max_workers=len(workers)) as pool:
    futures={pool.submit(render_one,s,workers[i%len(workers)]):s for i,s in enumerate(segments)}
    for fut in as_completed(futures):
        try:
            idx,files,status=fut.result(); results[idx]=files; print(f'Segment {idx}: {status} ->',files)
        except Exception as e:
            failures.append(str(e)); print('SEGMENT ERROR:',repr(e))
if failures: raise RuntimeError('Some segments failed. Fix the issue and rerun this cell; completed segments will resume.\n'+'\n'.join(failures))


In [ ]:
# 11. Final assembly and exact original audio mux
# Select the first MP4 produced per segment. If your workflow saves another format, convert it here.
video_parts=[]
for seg in segments:
    paths=completed(seg['index'])
    if not paths: raise RuntimeError(f'Missing completed segment {seg[\"index\"]}')
    vids=[p for p in paths if p.suffix.lower() in ('.mp4','.mkv','.mov','.webm')]
    if not vids: raise RuntimeError(f'No video file in segment {seg[\"index\"]}: {paths}')
    video_parts.append(sorted(vids)[0])
concat=WORK_DIR/'concat.txt'
concat.write_text(''.join("file '"+str(p.resolve()).replace("'","'\\''")+"'\\n" for p in video_parts))
joined=OUTPUT_DIR/'joined_video.mp4'
run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(concat),'-an','-c:v','libx264','-preset','veryfast','-crf','18','-pix_fmt','yuv420p',str(joined)])
final=OUTPUT_DIR/'Hindi_Poem_Final.mp4'
run(['ffmpeg','-y','-i',str(joined),'-i',str(audio_path),'-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-b:a','192k','-shortest','-movflags','+faststart',str(final)])
print('FINAL VIDEO:',final,'size:',round(final.stat().st_size/1024**2,2),'MiB')


In [ ]:
# 12. Optional download to the Kaggle notebook output area
from IPython.display import display, Video
display(Video(str(final),embed=True,width=800))
print('Output directory:',OUTPUT_DIR)


In [ ]:
# 13. Shutdown ComfyUI worker processes cleanly
for wkr in workers:
    try: wkr['proc'].terminate(); wkr['proc'].wait(timeout=20)
    except Exception: wkr['proc'].kill()
    try: wkr['log'].close()
    except Exception: pass
print('Workers stopped.')
